# 01 · Basic Transformer (encoder–decoder) trained from scratch — English → Hindi

Foundations of Generative AI (ICT4403) — Assignment 3

The original *Attention Is All You Need* architecture (Vaswani et al., 2017) is implemented with `torch.nn.Transformer` and
trained **from scratch** (no pretraining) on the 100k cleaned IITB training pairs prepared in `00_data_preparation.ipynb`.

* Tokenisation: a joint English+Hindi **SentencePiece BPE** model (16k vocabulary) trained on the same 100k pairs.
* Model: 4 encoder + 4 decoder layers, d_model 512, 8 heads, FFN 2048, dropout 0.1, sinusoidal positions,
  one embedding matrix shared by encoder input, decoder input and output projection.
* Training: AdamW (β = 0.9/0.98), 4k-step warm-up + inverse-sqrt decay, label smoothing 0.1, mixed precision,
  up to 20 epochs with early stopping on validation loss (patience 3).
* Decoding: greedy for the main results, temperature sampling for the temperature study.

**Run:** Kaggle/Colab with a GPU (T4 is enough) and internet on → *Run all* (~40 min).
Download `results_transformer.zip` at the end.

In [ ]:
!pip -q install sacrebleu sentencepiece

In [ ]:
import os, json, time, random, re
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run used only to test the code
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_URL = os.environ.get("DATA_URL", "https://raw.githubusercontent.com/UtkarshKumarJha/GenAI-IA3/main/data/")
load = lambda name: pd.read_json(DATA_URL + name, lines=True)
train, valid, test, probes = (load(f) for f in ("train.jsonl", "valid.jsonl", "test.jsonl", "probes.jsonl"))
temp_set = test.iloc[:200]          # subset used for the sampling-temperature study
TEMPS = [0.5, 1.0, 1.5]             # main results use greedy decoding (T -> 0)
if SMOKE:
    train, valid, test, temp_set, probes = train[:200], valid[:16], test[:8], test[:4], probes[:4]
print(f"train {len(train):,} | valid {len(valid)} | test {len(test)} | probes {len(probes)} | device {DEVICE}",
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
def save_jsonl(df, path):
    df.to_json(path, orient="records", lines=True, force_ascii=False)

def run_all(translate, tag, out):
    """Greedy test translations (timed), temperature samples, and probe translations -> out/"""
    os.makedirs(out, exist_ok=True)
    t0 = time.perf_counter()
    hyps = translate(test.en.tolist())
    infer_s = time.perf_counter() - t0
    save_jsonl(test.assign(hyp=hyps), f"{out}/test_preds.jsonl")
    import sacrebleu
    bleu = sacrebleu.corpus_bleu(hyps, [test.hi.tolist()], tokenize="intl").score
    chrf = sacrebleu.corpus_chrf(hyps, [test.hi.tolist()]).score
    print(f"[{tag}] BLEU {bleu:.2f} | chrF {chrf:.2f} | {infer_s:.1f}s for {len(test)} sentences")
    for en, ref, hyp in list(zip(test.en, test.hi, hyps))[:3]:
        print(f"  EN : {en}\n  REF: {ref}\n  HYP: {hyp}\n")
    save_jsonl(probes.assign(hyp=translate(probes.en.tolist())), f"{out}/probe_preds.jsonl")
    return dict(infer_time_s=infer_s, infer_sent_per_s=len(test) / infer_s, quick_bleu=bleu, quick_chrf=chrf)

def run_temperatures(translate_sampled, out):
    rows = []
    for T in TEMPS:
        torch.manual_seed(SEED)
        for en, ref, hyp in zip(temp_set.en, temp_set.hi, translate_sampled(temp_set.en.tolist(), T)):
            rows.append(dict(temperature=T, en=en, hi=ref, hyp=hyp))
    save_jsonl(pd.DataFrame(rows), f"{out}/temperature_preds.jsonl")

In [ ]:
import math, sentencepiece as spm
import torch.nn as nn, torch.nn.functional as F

CFG = dict(vocab=16000, d_model=512, heads=8, layers=4, ffn=2048, dropout=0.1, max_len=128,
           batch=64, lr=7e-4, warmup=4000, label_smoothing=0.1, epochs=20, patience=3)
if SMOKE: CFG.update(vocab=500, d_model=64, heads=2, layers=1, ffn=128, batch=16, warmup=10, epochs=1)
OUT = "results/transformer"; os.makedirs(OUT, exist_ok=True)

with open("spm_train.txt", "w") as f:
    f.write("\n".join(train.en.tolist() + train.hi.tolist()))
spm.SentencePieceTrainer.train(input="spm_train.txt", model_prefix="spm_enhi", vocab_size=CFG["vocab"],
                               model_type="bpe", character_coverage=1.0, pad_id=0, unk_id=1, bos_id=2, eos_id=3,
                               input_sentence_size=200_000, shuffle_input_sentence=True, minloglevel=2)
sp = spm.SentencePieceProcessor(model_file="spm_enhi.model")
PAD, BOS, EOS = 0, 2, 3
s = test.en[0]
print(sp.encode(s, out_type=str)[:25]); print(sp.encode(test.hi[0], out_type=str)[:25])
for col in ("en", "hi"):
    print(col, "avg tokens per sentence:", np.mean([len(x) for x in sp.encode(train[col][:5000].tolist())]).round(1))

## Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=1024):
        super().__init__()
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe = torch.zeros(max_len, d); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)
    def forward(self, x): return x + self.pe[: x.size(1)]

class Seq2SeqTransformer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d = c["d_model"]
        self.emb = nn.Embedding(c["vocab"], self.d, padding_idx=PAD)
        nn.init.normal_(self.emb.weight, 0, self.d ** -0.5)   # tied in/out embedding: keep initial logits ~unit scale
        self.pos = PositionalEncoding(self.d)
        self.drop = nn.Dropout(c["dropout"])
        self.tf = nn.Transformer(self.d, c["heads"], c["layers"], c["layers"], c["ffn"], c["dropout"], batch_first=True)
    def embed(self, x): return self.drop(self.pos(self.emb(x) * math.sqrt(self.d)))
    def encode(self, src):
        return self.tf.encoder(self.embed(src), src_key_padding_mask=src == PAD)
    def decode(self, tgt, memory, src):
        causal = torch.triu(torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool, device=tgt.device), 1)
        h = self.tf.decoder(self.embed(tgt), memory, tgt_mask=causal, tgt_key_padding_mask=tgt == PAD,
                            memory_key_padding_mask=src == PAD)
        return h @ self.emb.weight.T          # output projection tied to the shared embedding
    def forward(self, src, tgt): return self.decode(tgt, self.encode(src), src)

model = Seq2SeqTransformer(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f} M")

## Data pipeline and training

In [ ]:
def encode_pairs(df):
    src = sp.encode(df.en.tolist()); tgt = sp.encode(df.hi.tolist())
    return [(s[:CFG["max_len"]], [BOS] + t[:CFG["max_len"] - 2] + [EOS]) for s, t in zip(src, tgt)]

def pad(seqs):
    out = torch.full((len(seqs), max(map(len, seqs))), PAD, dtype=torch.long)
    for i, s in enumerate(seqs): out[i, : len(s)] = torch.tensor(s)
    return out

def batches(pairs, shuffle):
    idx = np.random.permutation(len(pairs)) if shuffle else np.arange(len(pairs))
    # bucket by length inside chunks of 100 batches -> less padding, still random
    chunk = CFG["batch"] * 100
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c: c + chunk], key=lambda i: len(pairs[i][0]))
        bs = [part[i: i + CFG["batch"]] for i in range(0, len(part), CFG["batch"])]
        if shuffle: random.shuffle(bs)
        for b in bs:
            yield pad([pairs[i][0] for i in b]).to(DEVICE), pad([pairs[i][1] for i in b]).to(DEVICE)

train_pairs, valid_pairs = encode_pairs(train), encode_pairs(valid)
crit = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=CFG["label_smoothing"])
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / CFG["warmup"], (CFG["warmup"] / (s + 1)) ** 0.5))
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_epoch(pairs, train_mode):
    model.train(train_mode); tot, n = 0.0, 0
    with torch.set_grad_enabled(train_mode):
        for src, tgt in batches(pairs, train_mode):
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(src, tgt[:, :-1])
                loss = crit(logits.float().reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            if train_mode:
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
            k = (tgt[:, 1:] != PAD).sum().item(); tot += loss.item() * k; n += k
    return tot / n

history, best, bad = [], float("inf"), 0
t0 = time.perf_counter()
for ep in range(1, CFG["epochs"] + 1):
    te = time.perf_counter()
    tr, va = run_epoch(train_pairs, True), run_epoch(valid_pairs, False)
    history.append(dict(epoch=ep, train_loss=tr, val_loss=va, val_ppl=math.exp(va), sec=time.perf_counter() - te))
    print(f"epoch {ep:2d} | train {tr:.3f} | val {va:.3f} (ppl {math.exp(va):.1f}) | {history[-1]['sec']:.0f}s")
    if va < best: best, bad = va, 0; torch.save(model.state_dict(), "transformer_best.pt")
    else:
        bad += 1
        if bad >= CFG["patience"]: print("early stopping"); break
train_time = time.perf_counter() - t0
model.load_state_dict(torch.load("transformer_best.pt"))
pd.DataFrame(history).to_csv(f"{OUT}/history.csv", index=False)
print(f"training time {train_time / 60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
plt.figure(figsize=(6, 3.5)); plt.plot(h.epoch, h.train_loss, label="train"); plt.plot(h.epoch, h.val_loss, label="validation")
plt.xlabel("epoch"); plt.ylabel("label-smoothed CE loss"); plt.title("Transformer training curve"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/training_curve.png", dpi=120); plt.show()

## Translation (greedy and temperature sampling)

In [ ]:
@torch.no_grad()
def translate(sentences, temperature=0.0, bs=64, max_new=200):
    model.eval(); out = []
    for i in range(0, len(sentences), bs):
        src = pad([s[:CFG["max_len"]] for s in sp.encode(sentences[i: i + bs])]).to(DEVICE)
        mem = model.encode(src)
        ys = torch.full((len(src), 1), BOS, dtype=torch.long, device=DEVICE)
        done = torch.zeros(len(src), dtype=torch.bool, device=DEVICE)
        for _ in range(max_new):
            logits = model.decode(ys, mem, src)[:, -1].float()
            logits[:, PAD] = logits[:, BOS] = -float("inf")
            nxt = (logits.argmax(-1) if temperature == 0
                   else torch.multinomial(F.softmax(logits / temperature, -1), 1).squeeze(1))
            nxt = nxt.masked_fill(done, PAD)
            ys = torch.cat([ys, nxt[:, None]], 1); done |= nxt == EOS
            if done.all(): break
        for row in ys[:, 1:].tolist():
            row = row[: row.index(EOS)] if EOS in row else row
            out.append(sp.decode([t for t in row if t != PAD]))
    return out

stats = dict(model="Transformer (from scratch)", architecture="encoder-decoder", adaptation="trained from scratch",
             params=n_params, trainable_params=n_params, train_pairs=len(train), train_time_s=train_time,
             epochs=len(history), best_val_loss=best, tokenizer=f"SentencePiece BPE {CFG['vocab']} (joint EN+HI)",
             avg_tokens_en=float(np.mean([len(x) for x in sp.encode(test.en.tolist())])),
             avg_tokens_hi=float(np.mean([len(x) for x in sp.encode(test.hi.tolist())])))
stats.update(run_all(translate, "Transformer", OUT))
run_temperatures(lambda xs, T: translate(xs, temperature=T), OUT)
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=1)
stats

In [ ]:
import shutil; shutil.make_archive("results_transformer", "zip", ".", OUT)   # download this file